# Fold-local SISSO/Boruta Y-randomization

This notebook runs and visualizes the response-permutation experiment requested during revision. The computational functions live in separate Python modules in this directory.

For the observed response and every shuffled response, the workflow creates five outer folds, fits the SISSO/Boruta feature generator using only each outer-training partition, reconstructs the selected expressions on the corresponding untouched outer-test partition, performs repeated inner-CV 1-SE LASSO selection with fold-local scaling, and produces one OOF prediction per observation.

## Methodological scope

SISSO/Boruta is rerun in every **outer-training fold**. It is not rerun again within each inner fold used to optimize the LASSO alpha. Consequently, the reported outer OOF performance is protected from feature-selection leakage into the held-out observations.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from nested_lasso import LassoConfig
from sisso_fold_features import SISSOConfig
from run_sisso_y_randomization import (
    ExperimentConfig,
    inspect_progress,
    load_fold_debug,
    load_experiment_results,
    load_input_table,
    run_experiment,
)
from plot_sisso_y_randomization import (
    feature_selection_frequency,
    plot_best_randomized_parity,
    plot_null_distributions,
    plot_top_feature_frequencies,
    save_figure,
)

## Configuration

Start with one or two shuffles to verify the complete installation and estimate runtime. The run is checkpointed after every outer fold, and increasing `N_SHUFFLES` later resumes the existing calculation.

In [ ]:
WORKFLOW_DIR = Path.cwd().resolve()
if not (WORKFLOW_DIR / "run_sisso_y_randomization.py").exists():
    raise RuntimeError(
        "Start Jupyter from the sisso_y_randomization directory before running."
    )

INPUT_FILE = (
    WORKFLOW_DIR.parent / "baseline_validation" / "training_set_base.csv"
)
LEGACY_SISSO_DIR = (
    WORKFLOW_DIR.parents[1] / "descriptor_generation" / "sisso"
)
OUTPUT_DIR = WORKFLOW_DIR / "results"

N_SHUFFLES = 2  # smoke test; increase to 200 or 1000 for reporting
RANDOM_STATE = 42

sisso_config = SISSOConfig(
    legacy_sisso_dir=LEGACY_SISSO_DIR,
    collinearity_cutoff=0.8,
    relative_filter_permutations=1000,
    boruta_percentile=75,
    boruta_max_iter=100,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=False,
    nonfinite_test_policy="raise",
)

lasso_config = LassoConfig(
    alphas=np.logspace(-4, -1, 100),
    n_bins=5,
    inner_splits=5,
    inner_repeats=4,
    random_state=RANDOM_STATE,
    max_iter=100_000,
)

config = ExperimentConfig(
    input_file=INPUT_FILE,
    output_dir=OUTPUT_DIR,
    legacy_sisso_dir=LEGACY_SISSO_DIR,
    n_shuffles=N_SHUFFLES,
    outer_splits=5,
    n_bins=5,
    random_state=RANDOM_STATE,
    resume=True,
    sisso=sisso_config,
    lasso=lasso_config,
)

print(f"Input: {INPUT_FILE}")
print(f"Legacy SISSO scripts: {LEGACY_SISSO_DIR}")
print(f"Results: {OUTPUT_DIR}")
print(f"Requested shuffles: {N_SHUFFLES}")

## Inspect the fixed linear input matrix

In [ ]:
input_table, base_features = load_input_table(INPUT_FILE)
print(f"Rows: {len(input_table)}")
print(f"Base descriptors: {len(base_features)}")
display(input_table.head())

## Run or resume the experiment

This is the expensive cell. The observed-response reference is evaluated first, followed by the requested permutations. Each completed outer fold is cached under `results/runs/`.

In [ ]:
result = run_experiment(config)

## Load existing results without fitting

After restarting the kernel, run the configuration cells and this cell to regenerate tables and figures without rerunning SISSO.

In [ ]:
result = load_experiment_results(OUTPUT_DIR)

## Intermediate progress and debugging

In [ ]:
progress = inspect_progress(OUTPUT_DIR)
display(progress)

In [ ]:
# Change these values to inspect a particular completed or active fold.
DEBUG_RESPONSE_ID = "observed"
DEBUG_OUTER_FOLD = 1

debug_record, debug_log = load_fold_debug(
    OUTPUT_DIR,
    DEBUG_RESPONSE_ID,
    DEBUG_OUTER_FOLD,
)
display(pd.Series(debug_record, name="value").to_frame())
print(debug_log[-5000:])  # final 5000 characters of the SISSO log

## Numerical summary

In [ ]:
display(result.empirical_significance.style.format(precision=4))

summary_columns = [
    "response_id",
    "mean_in_fold_r2",
    "mean_in_fold_mae",
    "oof_r2",
    "oof_mae",
    "alpha_1se_median",
    "selected_augmented_median",
    "nonzero_lasso_median",
]
display(result.response_summary[summary_columns].style.format(precision=4))

## OOF null distributions

In [ ]:
null_figure, null_axes = plot_null_distributions(
    result.response_summary,
    result.empirical_significance,
)
save_figure(null_figure, OUTPUT_DIR / "sisso_y_randomization_null_distributions")
plt.show()

## Best randomized-model parity plot

The displayed points are outer-fold predictions from the permutation with the highest OOF R².

In [ ]:
parity_figure, parity_axis = plot_best_randomized_parity(
    result.response_summary,
    result.predictions,
)
save_figure(parity_figure, OUTPUT_DIR / "best_randomized_model_parity")
plt.show()

## SISSO/Boruta selection stability for the observed response

In [ ]:
observed_feature_frequency = feature_selection_frequency(
    result.fold_metrics,
    response_kind="observed",
)
display(observed_feature_frequency.head(25).style.format({"selection_frequency": "{:.2f}"}))

if not observed_feature_frequency.empty:
    feature_figure, feature_axis = plot_top_feature_frequencies(
        result.fold_metrics,
        top_n=20,
    )
    save_figure(feature_figure, OUTPUT_DIR / "observed_feature_selection_frequency")
    plt.show()

## Fold-level audit table

In [ ]:
audit_columns = [
    "response_id",
    "outer_fold",
    "n_train",
    "n_test",
    "n_selected_augmented_features",
    "alpha_min",
    "alpha_1se",
    "n_nonzero",
    "in_fold_r2",
    "in_fold_mae",
    "outer_test_r2",
    "outer_test_mae",
    "elapsed_seconds",
]
display(result.fold_metrics[audit_columns].style.format(precision=4))